# 02 · Counterfactual: synthetic Myanmar

What would Myanmar have looked like without the February 2021 coup? No country gives a clean comparison on its own. The **synthetic control method** builds one: a weighted blend of peer countries that didn't have a 2021 rupture, with the weights chosen so the blend tracks real Myanmar as closely as possible *before* 2021. After 2021, the gap between real and synthetic Myanmar is the estimated effect.

This notebook is a walkthrough; the logic lives in `src/amber/modeling/synthetic_control.py` and `make sc` produces the same tables and charts.

**Prerequisites:** `make panel` and `make index`.

In [ ]:
%matplotlib inline
import logging
from dataclasses import replace

import pandas as pd

from amber import config, counterfactual, figures
from amber.modeling import synthetic_control as sc

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.precision", 3)

panel, index = counterfactual.load_inputs(config.SC_OUTCOMES)
settings = sc.SCSettings()
settings

## 1 · The fit

The features are Myanmar's outcome in each year from 2011 to 2020, z-scored across Myanmar and the six donors. The weights solve

$$\min_w \; \lVert x_1 - X_0 w \rVert^2 \quad \text{s.t.} \quad w_j \ge 0,\; \textstyle\sum_j w_j = 1$$

The constraint is the point: synthetic Myanmar has to sit *inside* the range of real donors, so the counterfactual interpolates between countries that exist rather than extrapolating past them. The first outcome is real GDP per capita.

In [ ]:
gdp = sc.run("NY.GDP.PCAP.KD", panel=panel, index=index, settings=settings)
base = gdp.base

pd.Series(base.weights, name="weight").to_frame().assign(
    donor=lambda d: d.index.map(config.COUNTRIES)
)

In [ ]:
pd.Series(
    {
        "pre-RMSE ($)": base.pre_rmse,
        "pre-RMSE / pre-period level": base.pre_rmse_share,
        "post-RMSE ($)": base.post_rmse,
        "post/pre ratio": base.rmse_ratio,
        "effective donors": base.n_effective_donors,
        "2024 gap ($)": base.gap[2024],
        "2024 gap (% of synthetic)": base.gap[2024] / base.synthetic[2024],
    }
)

In [ ]:
figures.actual_vs_synthetic_figure(gdp)

## 2 · Is the gap unusual?

**In-space placebo.** Refit each donor as if *it* had been treated in 2021, matched against the other donors, never Myanmar. If Myanmar's gap is real, its post/pre RMSE ratio should stand out from theirs. The pseudo p-value is the share of units whose ratio is at least Myanmar's.

With 7 units the smallest possible p-value is **1/7 ≈ 0.14**. The test can rank Myanmar first, but it can't produce p < 0.05 however large the effect is.

In [ ]:
placebos = gdp.placebo_space
(
    placebos.ratios.sort_values(ascending=False)
    .to_frame()
    .assign(unit=lambda d: d.index.map(config.COUNTRIES)),
    placebos.p_value,
)

In [ ]:
figures.placebo_gaps_figure(gdp)

## 3 · Robustness

- **In-time placebo:** pretend the coup happened in 2017 and use only pre-2021 data. A good design shows a gap of about 0 from 2017 to 2019. 2020 is Myanmar's own anomaly (see §4).
- **Leave-one-out:** drop each donor that carries weight and refit. If one donor drives the result, dropping it moves the synthetic a lot.

In [ ]:
gdp.placebo_time.gap.loc[2017:2020].rename("in-time placebo gap").to_frame()

In [ ]:
pd.DataFrame(
    {
        f"without {config.COUNTRIES[d]}": refit.gap.loc[2021:]
        for d, refit in gdp.leave_one_out.items()
    }
).assign(base=base.gap.loc[2021:])

## 4 · The 2020 choice

The fit window keeps 2020. COVID hit the donors too, so matching on 2020 controls for it. But Myanmar's WDI 2020 (fiscal October 2019 to September 2020, entirely before the coup) records −9.1% growth, below every donor, so no convex combination can reach it. Here is how much that one year matters, along with the rebased variant (2011 = 100):

In [ ]:
variants = {
    "fit to 2020 (default)": settings,
    "fit to 2019": replace(settings, pre_end=2019),
    "rebased": replace(settings, rebase=True),
}
rows = {}
for label, s in variants.items():
    r = sc.run("NY.GDP.PCAP.KD", panel=panel, index=index, settings=s)
    rows[label] = {
        "weights": {
            config.COUNTRIES[d]: round(w, 2) for d, w in r.base.weights.items() if w >= 0.01
        },
        "pre-RMSE share": r.base.pre_rmse_share,
        "2024 gap (% of synthetic)": r.base.gap[2024] / r.base.synthetic[2024],
        "p-value": r.p_value,
    }
pd.DataFrame(rows).T

## 5 · The combined development index

The same method, applied to the combined index from `make index`. Watch the pre-period: Myanmar starts **below every donor** (2011–2013), so no convex combination can reach it. The pre-fit misses by about a quarter of Myanmar's own level, and the post-2021 gap can't be read as an effect. The verdict is data, not only a caption: `credible` is false here and in `sc_metrics`.

In [ ]:
combined = sc.run("combined", panel=panel, index=index, settings=settings)
combined.base.pre_rmse_share, combined.base.credible, combined.p_value

In [ ]:
figures.actual_vs_synthetic_figure(combined)

## Reading the results

- **GDP per capita:** a close pre-fit (about 3% of level) and a large, growing gap after 2021. Myanmar ranks first of 7 in the placebo test, the strongest result 7 units can give. The gap stays negative when any donor is dropped. How big it is depends on which donors are used and on the 2020 choice.
- **Combined index:** the donor pool can't reproduce Myanmar's pre-2021 path, so this outcome is inconclusive rather than evidence of no effect.